# Week 2: Convolutional Networks and the UNet — From Scratch
**Diffusion Models from Scratch — Seasons of Code 2026**

### Deliverables Checklist
- [x] Modular building blocks: `DoubleConv`, `Down`, `Up`
- [x] Configurable depth and channel count (passed via constructor)
- [x] Skip connections correctly implemented (concatenation, not addition)
- [x] Trained on MNIST denoising task
- [x] Visualization: noisy input → denoised output every N epochs
- [ ] Code pushed to GitHub with updated README

---

## 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Understanding Convolutions

A **convolution** slides a small filter (e.g. 3×3) over an image and computes a weighted sum at each position — detecting local patterns regardless of where they appear.

Key parameters:
- `kernel_size`: size of filter (3×3 standard)
- `stride`: step size of the filter
- `padding=1`: pads border so output stays same H×W
- `out_channels`: number of different filters (each detects a different feature)

In [ ]:
transform = transforms.ToTensor()
demo_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
demo_img, _ = demo_dataset[0]
conv_demo = nn.Conv2d(1, 8, kernel_size=3, padding=1)
with torch.no_grad():
    feat = conv_demo(demo_img.unsqueeze(0))
fig, axes = plt.subplots(1, 9, figsize=(16, 2))
axes[0].imshow(demo_img.squeeze(), cmap='gray')
axes[0].set_title('Original', fontsize=8)
axes[0].axis('off')
for i in range(8):
    axes[i + 1].imshow(feat[0, i].numpy(), cmap='viridis')
    axes[i + 1].set_title(f'Filter {i + 1}', fontsize=8)
    axes[i + 1].axis('off')
plt.suptitle('One Conv2d layer: 8 learned feature maps from a single digit', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Input: {demo_img.unsqueeze(0).shape}  →  Feature maps: {feat.shape}')

## 2. Modular Building Blocks

Three reusable modules — `DoubleConv`, `Down`, `Up` — compose the full UNet.

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        dh = skip.shape[2] - x.shape[2]
        dw = skip.shape[3] - x.shape[3]
        x = F.pad(x, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)
x = torch.randn(2, 1, 28, 28)
s = DoubleConv(1, 64)(x)
d = Down(64, 128)(s)
u = Up(128 + 64, 64)(d, s)
print(f'DoubleConv output : {s.shape}')
print(f'Down output       : {d.shape}')
print(f'Up+skip output    : {u.shape}   ← matches original H×W')

## 3. Full UNet — Configurable Depth and Channel Count

The UNet constructor accepts:
- `in_channels` / `out_channels` — for any image type
- `base_channels` — controls width (number of feature maps)
- `depth` — controls how many Down/Up stages (configurable depth)

```
Input
 ↓
enc[0] ─────────────────────────────────── skip[0] ──┐
 ↓                                                    │
enc[1] ──────────────────────── skip[1] ──┐           │
 ↓                                        │           │
enc[2] ──────────── skip[2] ──┐           │           │
 ↓                            │           │           │
bottleneck                    │           │           │
 ↓                            │           │           │
dec[2] ← concat(skip[2]) ─────┘           │           │
 ↓                                        │           │
dec[1] ← concat(skip[1]) ─────────────────┘           │
 ↓                                                    │
dec[0] ← concat(skip[0]) ─────────────────────────────┘
 ↓
out_conv → same H×W as input
```

In [ ]:
class UNet(nn.Module):
    

    def __init__(self, in_channels=1, out_channels=1, base_channels=64, depth=3):
        super().__init__()
        self.depth = depth
        self.enc_blocks = nn.ModuleList()
        self.enc_blocks.append(DoubleConv(in_channels, base_channels))
        ch = base_channels
        for _ in range(depth - 1):
            self.enc_blocks.append(Down(ch, ch * 2))
            ch *= 2
        self.bottleneck = Down(ch, ch * 2)
        ch *= 2
        self.dec_blocks = nn.ModuleList()
        for _ in range(depth):
            self.dec_blocks.append(Up(ch + ch // 2, ch // 2))
            ch //= 2
        self.out_conv = nn.Conv2d(ch, out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc in self.enc_blocks:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        for dec, skip in zip(self.dec_blocks, reversed(skips)):
            x = dec(x, skip)
        return self.out_conv(x)
for depth in [2, 3, 4]:
    m = UNet(depth=depth)
    dummy = torch.randn(2, 1, 32, 32)
    out = m(dummy)
    params = sum((p.numel() for p in m.parameters()))
    print(f'depth={depth}  input={tuple(dummy.shape)}  output={tuple(out.shape)}  params={params:,}')

In [ ]:
def shape_trace(model, input_shape=(2, 1, 28, 28)):
    shapes, hooks = ({}, [])
    named = [*[(f'enc[{i}]', b) for i, b in enumerate(model.enc_blocks)], ('bottleneck', model.bottleneck), *[(f'dec[{i}]', b) for i, b in enumerate(model.dec_blocks)], ('out_conv', model.out_conv)]
    for name, layer in named:
        hooks.append(layer.register_forward_hook(lambda m, i, o, n=name: shapes.__setitem__(n, tuple(o.shape))))
    with torch.no_grad():
        model(torch.randn(*input_shape))
    for h in hooks:
        h.remove()
    print(f'{'Layer':<15} Output Shape')
    print('-' * 38)
    for k, v in shapes.items():
        print(f'{k:<15} {v}')
model = UNet(in_channels=1, out_channels=1, base_channels=64, depth=3).to(device)
print(f'Total params: {sum((p.numel() for p in model.parameters())):,}\n')
shape_trace(model)

## 4. Denoising Dataset (MNIST)

In [ ]:
class NoisyMNIST(Dataset):
    

    def __init__(self, root, train=True, noise_std=0.4, download=True):
        self.ds = torchvision.datasets.MNIST(root, train=train, download=download, transform=transforms.ToTensor())
        self.noise_std = noise_std

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        clean, _ = self.ds[idx]
        noisy = torch.clamp(clean + torch.randn_like(clean) * self.noise_std, 0, 1)
        return (noisy, clean)
NOISE_STD = 0.4
BATCH_SIZE = 128
train_ds = NoisyMNIST('./data', train=True, noise_std=NOISE_STD)
val_ds = NoisyMNIST('./data', train=False, noise_std=NOISE_STD)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
noisy_b, clean_b = next(iter(val_loader))
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(10):
    axes[0, i].imshow(noisy_b[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    axes[1, i].imshow(clean_b[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Noisy (input)', fontsize=10, rotation=90, labelpad=40, va='center')
axes[1, 0].set_ylabel('Clean (target)', fontsize=10, rotation=90, labelpad=40, va='center')
plt.suptitle(f'Denoising Task — noise_std={NOISE_STD}', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Training with Visualization Every N Epochs

We visualize noisy → denoised every `VIZ_EVERY` epochs so we can watch the model improve.

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for noisy, clean in loader:
        noisy, clean = (noisy.to(device), clean.to(device))
        optimizer.zero_grad()
        loss = criterion(model(noisy), clean)
        loss.backward()
        optimizer.step()
        total += loss.item() * noisy.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total = 0.0
    for noisy, clean in loader:
        noisy, clean = (noisy.to(device), clean.to(device))
        total += criterion(model(noisy), clean).item() * noisy.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def visualize_epoch(model, noisy_fixed, clean_fixed, epoch):
    
    model.eval()
    denoised = model(noisy_fixed.to(device)).cpu().clamp(0, 1)
    n = 8
    fig, axes = plt.subplots(3, n, figsize=(16, 5))
    for i in range(n):
        axes[0, i].imshow(noisy_fixed[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[0, i].axis('off')
        axes[1, i].imshow(denoised[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[1, i].axis('off')
        axes[2, i].imshow(clean_fixed[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[2, i].axis('off')
    axes[0, 0].set_ylabel('Noisy Input', fontsize=9, rotation=90, labelpad=45, va='center')
    axes[1, 0].set_ylabel('UNet Output', fontsize=9, rotation=90, labelpad=45, va='center')
    axes[2, 0].set_ylabel('Clean Target', fontsize=9, rotation=90, labelpad=45, va='center')
    plt.suptitle(f'Noisy → Denoised  (Epoch {epoch})', fontsize=12)
    plt.tight_layout()
    plt.show()
fixed_noisy, fixed_clean = next(iter(val_loader))
EPOCHS = 12
VIZ_EVERY = 3
LR = 0.001
model = UNet(in_channels=1, out_channels=1, base_channels=64, depth=3).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()
train_losses, val_losses = ([], [])
print(f'Training UNet  |  depth=3  |  base_channels=64  |  {EPOCHS} epochs')
print(f'{'Epoch':<8}{'Train MSE':<14}{'Val MSE'}')
print('-' * 36)
for epoch in range(1, EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer, criterion)
    vl = val_epoch(model, val_loader, criterion)
    scheduler.step()
    train_losses.append(tr)
    val_losses.append(vl)
    print(f'{epoch:<8}{tr:<14.6f}{vl:.6f}')
    if epoch % VIZ_EVERY == 0 or epoch == 1:
        visualize_epoch(model, fixed_noisy, fixed_clean, epoch)
print('Training complete!')

## 6. Training Curves & Final Results

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train MSE', marker='o')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Val MSE', marker='s')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('UNet Training — MNIST Denoising')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final Val MSE: {val_losses[-1]:.6f}')

## 7. Checkpoint Question — Skip Connections

> **"Why are skip connections in a UNet important? What happens to the gradients and information flow without them?"**

### Answer

The UNet encoder **compresses** spatial information — a 28×28 image shrinks to ~4×4 at the bottleneck. High-frequency spatial details (edges, exact pixel positions, textures) are progressively discarded by pooling.

The decoder must reconstruct full-resolution output from this compressed code. Without skip connections, it must hallucinate all the fine-grained detail it cannot see — producing **blurry outputs**.

**Skip connections (via concatenation)** route the encoder's feature maps directly to the matching decoder level:
- The decoder gets **full access** to early features before they were pooled away
- The concatenation doubles the channel count, so the decoder has both "where am I globally" (from bottleneck) and "what fine detail is here" (from skip) simultaneously

**Gradient flow**: Without skips, gradients must traverse the full encoder↔bottleneck↔decoder path — a long chain prone to vanishing gradients. Skips are shortcuts that carry strong gradients directly to early encoder layers, making training faster and more stable.

We prove this empirically below by training with and without skip connections.

In [ ]:
class UNetNoSkip(nn.Module):
    

    def __init__(self, in_channels=1, out_channels=1, base_channels=64, depth=3):
        super().__init__()
        self.depth = depth
        self.enc_blocks = nn.ModuleList()
        self.enc_blocks.append(DoubleConv(in_channels, base_channels))
        ch = base_channels
        for _ in range(depth - 1):
            self.enc_blocks.append(Down(ch, ch * 2))
            ch *= 2
        self.bottleneck = Down(ch, ch * 2)
        ch *= 2
        self.dec_blocks = nn.ModuleList()
        for _ in range(depth):
            self.dec_blocks.append(nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), DoubleConv(ch, ch // 2)))
            ch //= 2
        self.out_conv = nn.Conv2d(ch, out_channels, kernel_size=1)

    def forward(self, x):
        for enc in self.enc_blocks:
            x = enc(x)
        x = self.bottleneck(x)
        for dec in self.dec_blocks:
            x = dec(x)
        return self.out_conv(x)

def train_model(build_fn, epochs=8):
    m = build_fn().to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    crit = nn.MSELoss()
    history = []
    for ep in range(1, epochs + 1):
        tr = train_epoch(m, train_loader, opt, crit)
        vl = val_epoch(m, val_loader, crit)
        history.append((tr, vl))
        print(f'  ep {ep}/{epochs}  train={tr:.5f}  val={vl:.5f}')
    return (m, history)
COMPARE_EPOCHS = 8
print('=== WITH skip connections ===')
m_skip, h_skip = train_model(lambda: UNet())
print('\n=== WITHOUT skip connections ===')
m_noskip, h_noskip = train_model(lambda: UNetNoSkip())

In [ ]:
skip_v = [h[1] for h in h_skip]
noskip_v = [h[1] for h in h_noskip]
plt.figure(figsize=(8, 4))
plt.plot(range(1, COMPARE_EPOCHS + 1), skip_v, 'o-', color='royalblue', label='With Skip (UNet)')
plt.plot(range(1, COMPARE_EPOCHS + 1), noskip_v, 's--', color='tomato', label='Without Skip')
plt.xlabel('Epoch')
plt.ylabel('Validation MSE')
plt.title('Skip Connections Matter — Val Loss Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
pct = (noskip_v[-1] - skip_v[-1]) / noskip_v[-1] * 100
print(f'Final val MSE  WITH skip:    {skip_v[-1]:.6f}')
print(f'Final val MSE  WITHOUT skip: {noskip_v[-1]:.6f}')
print(f'Skip connections give {pct:.1f}% lower validation loss')

In [ ]:
m_skip.eval()
m_noskip.eval()
nb, cb = next(iter(val_loader))
with torch.no_grad():
    out_skip = m_skip(nb.to(device)).cpu().clamp(0, 1)
    out_noskip = m_noskip(nb.to(device)).cpu().clamp(0, 1)
n = 8
fig, axes = plt.subplots(4, n, figsize=(16, 8))
labels = ['Noisy Input', 'With Skip (UNet)', 'Without Skip', 'Clean Target']
rows = [nb, out_skip, out_noskip, cb]
for ri, (data, lbl) in enumerate(zip(rows, labels)):
    for ci in range(n):
        axes[ri, ci].imshow(data[ci].squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
        axes[ri, ci].axis('off')
    axes[ri, 0].set_ylabel(lbl, fontsize=9, rotation=90, labelpad=50, va='center')
plt.suptitle('Visual Impact of Skip Connections', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 8. Stretch Goal — Attention at the Bottleneck

Modern diffusion UNets insert self-attention at the bottleneck so every spatial position can attend to every other. This helps with global coherence.

In [ ]:
class SelfAttention2D(nn.Module):
    

    def __init__(self, channels, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.attn = nn.MultiheadAttention(channels, heads, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, H * W).permute(0, 2, 1)
        h, _ = self.attn(h, h, h)
        return x + h.permute(0, 2, 1).view(B, C, H, W)

class UNetWithAttention(nn.Module):
    

    def __init__(self, in_channels=1, out_channels=1, base_channels=64, depth=3):
        super().__init__()
        c = base_channels
        self.enc_blocks = nn.ModuleList([DoubleConv(in_channels, c)])
        ch = c
        for _ in range(depth - 1):
            self.enc_blocks.append(Down(ch, ch * 2))
            ch *= 2
        self.bottleneck = Down(ch, ch * 2)
        ch *= 2
        self.attention = SelfAttention2D(ch)
        self.dec_blocks = nn.ModuleList()
        for _ in range(depth):
            self.dec_blocks.append(Up(ch + ch // 2, ch // 2))
            ch //= 2
        self.out_conv = nn.Conv2d(ch, out_channels, 1)

    def forward(self, x):
        skips = []
        for enc in self.enc_blocks:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        x = self.attention(x)
        for dec, skip in zip(self.dec_blocks, reversed(skips)):
            x = dec(x, skip)
        return self.out_conv(x)
attn_model = UNetWithAttention().to(device)
with torch.no_grad():
    out = attn_model(torch.randn(2, 1, 28, 28).to(device))
print(f'UNet+Attention output: {out.shape}')
print(f'Params: {sum((p.numel() for p in attn_model.parameters())):,}')

## 9. Summary — Connection to Diffusion Models

| Component | Built This Week | Used in Diffusion |
|---|---|---|
| `DoubleConv` | ✅ | Feature extraction at every level |
| `Down` / `Up` | ✅ | Compress → expand spatial resolution |
| Skip connections | ✅ (concat) | Preserve spatial detail for denoising |
| Self-attention | ✅ (stretch) | Global coherence in latent diffusion |
| Configurable depth | ✅ | Scale capacity to dataset size |

**In Week 4**, this exact UNet gets one addition: a **timestep embedding** injected at every level, so the model learns to denoise at a specific noise level `t`. Every block you built here carries over directly.

In [ ]:
torch.save(model.state_dict(), 'week2_unet.pth')
print('Checkpoint saved: week2_unet.pth')
print(f'Final val MSE: {val_losses[-1]:.6f}')